# Phase 2: Data Preprocessing Pipeline (Training Set Only)
## CICIoT2023 Network Intrusion Detection

> **Lưu ý quan trọng từ người dùng:**
> File `plots/train.csv` (hoặc `train.csv`) **chỉ là tập Train** (dùng riêng cho huấn luyện). 
> Tập `validation.csv` và `test.csv` sẽ được cung cấp sau, do đó quy trình này **KHÔNG phân chia tập dữ liệu train/test trên file train.csv**.
> Toàn bộ pipeline tiền xử lý (Imputer, Zero-Variance filter, Feature Engineering, Label Encoder, RobustScaler) được fit trên tập Train và đóng gói vào `models/preprocessor.joblib` để sẵn sàng áp dụng trực tiếp lên `validation.csv` và `test.csv` khi có mà không lo rò rỉ dữ liệu (data leakage).

**Nhiệm vụ theo `PLAN.md` (Phase 2):**
1. **Xử lý Missing & Infinite (Task 2.1)**: Thay `inf`, `-inf` bằng `NaN`, tính `median` của từng cột và lưu lại để điền khuyết cho cả Train và Test/Validation sau này.
2. **Loại bỏ đặc trưng hằng số (Task 2.2)**: Tìm và loại bỏ các cột có phương sai bằng 0 (`variance = 0.0`), lưu danh sách cột bị loại.
3. **Feature Engineering chuyên sâu cho mạng IoT (Task 2.3)**:
   - Tỉ lệ packet & flow rate (`avg_packet_size`, `rate_ratio`, `drate_ratio`, `duration_ratio`, `header_to_payload_ratio`)
   - Flag aggregates (`total_flags`, `total_counts`, `syn_ack_ratio`, `rst_syn_ratio`)
   - Protocol groupings (`web_traffic`, `transport_traffic`, `network_mgmt`, `remote_access`, `is_encrypted`)
4. **Phân nhóm nhãn phân cấp (Task 2.4)**:
   - 35 Lớp chi tiết (`label`)
   - 8 Nhóm tấn công cấp cao (`attack_category`: DDoS, DoS, Mirai, Recon, Spoofing, Web, BruteForce, Malware, Benign)
   - Nhãn nhị phân (`is_attack`: 0 = Benign, 1 = Attack)
5. **Encoding nhãn & Chuẩn hóa đặc trưng (Task 2.5)**:
   - `LabelEncoder` cho nhãn tấn công và nhóm tấn công
   - `RobustScaler` (sử dụng Median & IQR, chống nhiễu từ các đợt spike lưu lượng mạng) fit trên toàn bộ tập Train
6. **Lưu trữ Deliverables & Hàm chuyển đổi tương lai**:
   - `X_train.pkl`, `y_train.pkl`, `data/X_train.parquet`
   - `models/preprocessor.joblib`, `models/label_mapping.json`
   - Cung cấp sẵn hàm `transform_val_or_test()` để tiền xử lý `validation.csv` và `test.csv` chỉ với 1 dòng lệnh.


In [ ]:
# ==============================================================================
# Cell 1: Import Libraries and Setup Configuration
# ==============================================================================
import os
import gc
import json
import warnings
import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import LabelEncoder, RobustScaler

warnings.filterwarnings('ignore')

# Cấu hình hiển thị pandas
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 1000)

# Tạo các thư mục lưu trữ nếu chưa có
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)

# -----------------------------------------------------------------------------
# Cấu hình dữ liệu:
# - SAMPLE_SIZE = None: Xử lý TOÀN BỘ tập Train (~5.5 triệu dòng)
# - SAMPLE_SIZE = 500_000 (hoặc 1_000_000): Dành cho máy có RAM <= 8GB nếu muốn chạy thử nhanh
# -----------------------------------------------------------------------------
SAMPLE_SIZE = None  # Đặt 500_000 nếu muốn test nhanh trên máy RAM 8GB, hoặc None cho toàn bộ train.csv
RANDOM_STATE = 42

print("Cấu hình môi trường thành công!")
print(f"Chế độ dữ liệu Train: {'TOÀN BỘ tập train.csv (~5.5M dòng)' if SAMPLE_SIZE is None else f'Lấy mẫu {SAMPLE_SIZE:,} dòng'}")
print("Ghi chú: train.csv được dùng 100% làm Training Set (không tách train/test nội bộ).")


## Bước 1: Tải dữ liệu Training Set tối ưu hóa RAM
Tự động tìm kiếm file `train.csv` (ưu tiên `plots/train.csv` hoặc symlink `train.csv`) và tải với kiểu dữ liệu `float32` để giảm 50% dung lượng RAM so với `float64` mặc định.


In [ ]:
# ==============================================================================
# Cell 2: Tải dữ liệu Training Set với Auto-Path Resolution & Optimized Dtypes
# ==============================================================================
candidate_paths = [
    'plots/train.csv',
    'train.csv',
    '../train.csv',
    '/content/drive/MyDrive/do_an/train.csv'
]

data_path = None
for p in candidate_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError(f"Không tìm thấy file train.csv trong các đường dẫn: {candidate_paths}")

file_size_mb = os.path.getsize(data_path) / 1024**2
print(f"Đang tải tập Training từ: {data_path} (Dung lượng: {file_size_mb:.2f} MB)...")

# Định nghĩa dtype float32 để tiết kiệm bộ nhớ
dtype_dict = {
    'flow_duration': 'float32', 'Header_Length': 'float32', 'Protocol Type': 'float32',
    'Duration': 'float32', 'Rate': 'float32', 'Srate': 'float32', 'Drate': 'float32',
    'fin_flag_number': 'float32', 'syn_flag_number': 'float32', 'rst_flag_number': 'float32',
    'psh_flag_number': 'float32', 'ack_flag_number': 'float32', 'ece_flag_number': 'float32',
    'cwr_flag_number': 'float32', 'ack_count': 'float32', 'syn_count': 'float32',
    'fin_count': 'float32', 'urg_count': 'float32', 'rst_count': 'float32',
    'HTTP': 'float32', 'HTTPS': 'float32', 'DNS': 'float32', 'Telnet': 'float32',
    'SMTP': 'float32', 'SSH': 'float32', 'IRC': 'float32', 'TCP': 'float32',
    'UDP': 'float32', 'DHCP': 'float32', 'ARP': 'float32', 'ICMP': 'float32',
    'IPv': 'float32', 'LLC': 'float32', 'Tot sum': 'float32', 'Min': 'float32',
    'Max': 'float32', 'AVG': 'float32', 'Std': 'float32', 'Tot size': 'float32',
    'IAT': 'float32', 'Number': 'float32', 'Magnitue': 'float32', 'Radius': 'float32',
    'Covariance': 'float32', 'Variance': 'float32', 'Weight': 'float32',
    'label': 'object'
}

if SAMPLE_SIZE is not None:
    df_train = pd.read_csv(data_path, dtype=dtype_dict, nrows=SAMPLE_SIZE)
else:
    df_train = pd.read_csv(data_path, dtype=dtype_dict)

gc.collect()

ram_usage_mb = df_train.memory_usage(deep=True).sum() / 1024**2
print(f"Tải dữ liệu Training thành công!")
print(f"Kích thước: {df_train.shape[0]:,} dòng x {df_train.shape[1]} cột")
print(f"Bộ nhớ RAM chiếm dụng: {ram_usage_mb:.2f} MB")
print(f"Số lượng nhãn tấn công độc nhất trong tập Train: {df_train['label'].nunique()}")


## Bước 2: Xử lý Missing & Infinite Values (Task 2.1)
Trong dữ liệu mạng, các phép chia thời gian bằng 0 (`Duration`, `flow_duration`) thường sinh ra giá trị vô cùng (`inf`, `-inf`).
- Thay thế toàn bộ `inf` và `-inf` bằng `NaN`.
- Tính giá trị trung vị (`median`) của từng cột số trên tập Train, điền khuyết cho tập Train.
- **Quan trọng:** Lưu lại dictionary các giá trị `imputation_medians` này để điền khuyết cho `validation.csv` và `test.csv` sau này theo đúng chuẩn (tránh rò rỉ dữ liệu).


In [ ]:
# ==============================================================================
# Cell 3: Xử lý giá trị Missing & Infinite (Task 2.1)
# ==============================================================================
print("="*60)
print("TASK 2.1: XỬ LÝ MISSING & INFINITE VALUES TRÊN TẬP TRAIN")
print("="*60)

numeric_cols = df_train.select_dtypes(include=['float32', 'float64']).columns.tolist()

# 1. Phát hiện và thay thế inf bằng NaN
inf_counts = {}
for col in numeric_cols:
    count_inf = np.isinf(df_train[col]).sum()
    if count_inf > 0:
        inf_counts[col] = count_inf

if inf_counts:
    print(f"Phát hiện giá trị vô cùng (inf) trong {len(inf_counts)} cột:")
    for col, count in inf_counts.items():
        print(f"  - {col}: {count:,} ({count/len(df_train)*100:.3f}%)")
    print("Thay thế inf/-inf bằng NaN...")
    df_train.replace([np.inf, -np.inf], np.nan, inplace=True)
else:
    print("Không phát hiện giá trị inf nào.")

# 2. Tính median trên tập Train và điền khuyết
imputation_medians = {}
nan_counts = df_train.isnull().sum()
cols_with_nan = nan_counts[nan_counts > 0]

print("\nTính toán median cho toàn bộ cột số để chuẩn bị pipeline...")
for col in numeric_cols:
    med = float(df_train[col].median())
    imputation_medians[col] = med

if len(cols_with_nan) > 0:
    print(f"Các cột có NaN trong tập Train ({len(cols_with_nan)} cột):")
    for col, count in cols_with_nan.items():
        print(f"  - {col}: {count:,} ({count/len(df_train)*100:.3f}%)")
    
    print("Điền khuyết bằng median của tập Train...")
    for col in cols_with_nan.index:
        if col in numeric_cols:
            df_train[col].fillna(imputation_medians[col], inplace=True)
            print(f"  -> {col} đã điền với median = {imputation_medians[col]:.4f}")
else:
    print("Tập Train không có giá trị NaN nào ban đầu.")

# 3. Loại bỏ dòng có nhãn label bị null nếu có
if df_train['label'].isnull().sum() > 0:
    dropped_count = df_train['label'].isnull().sum()
    df_train.dropna(subset=['label'], inplace=True)
    print(f"Đã loại {dropped_count} dòng có label bị trống.")

# Kiểm tra đảm bảo 0 NaN còn sót lại
total_nan = df_train.isnull().sum().sum()
print(f"\nKiểm tra lại: Tổng số NaN còn lại trong tập Train = {total_nan}")
assert total_nan == 0, "Lỗi: Vẫn còn giá trị NaN!"
print("Dữ liệu tập Train đã sạch 100% không còn missing/infinite values.")


## Bước 3: Loại bỏ đặc trưng hằng số (Task 2.2)
Các đặc trưng có phương sai bằng 0 (`variance = 0.0`) không mang thông tin phân lớp.
- Kiểm tra phương sai của các cột số trên tập Train.
- Phát hiện các cột hằng số và loại bỏ.
- Lưu lại danh sách `dropped_zero_var_cols` vào preprocessor để khi nạp `test.csv` / `validation.csv`, hệ thống tự động loại đúng các cột tương ứng.


In [ ]:
# ==============================================================================
# Cell 4: Loại bỏ đặc trưng hằng số (Task 2.2)
# ==============================================================================
print("="*60)
print("TASK 2.2: LOẠI BỎ ĐẶC TRƯNG HẰNG SỐ")
print("="*60)

feature_cols = [c for c in df_train.columns if c != 'label']

# 1. Phân tích phương sai
variances = df_train[feature_cols].var()
dropped_zero_var_cols = variances[variances == 0].index.tolist()

# 2. Phân tích cận hằng số (>99.95% cùng giá trị)
near_constant_cols = []
for col in feature_cols:
    if col not in dropped_zero_var_cols:
        top_freq = df_train[col].value_counts(normalize=True, dropna=False).iloc[0]
        if top_freq >= 0.9995:
            near_constant_cols.append((col, top_freq))

print(f"Số cột hằng số tuyệt đối (variance = 0.0): {len(dropped_zero_var_cols)}")
for col in dropped_zero_var_cols:
    print(f"  - {col} (variance = 0.0)")

print(f"\nSố cột cận hằng số (>99.95% trùng giá trị): {len(near_constant_cols)}")
for col, freq in near_constant_cols:
    print(f"  - {col}: {freq*100:.3f}% trùng")

# Loại bỏ các cột phương sai bằng 0
if dropped_zero_var_cols:
    df_train.drop(columns=dropped_zero_var_cols, inplace=True)
    print(f"\nĐã loại bỏ {len(dropped_zero_var_cols)} cột hằng số khỏi tập Train.")
else:
    print("\nKhông có cột hằng số tuyệt đối nào cần loại.")

print(f"Số lượng đặc trưng còn lại: {len(df_train.columns) - 1}")


## Bước 4: Feature Engineering chuyên sâu cho IoT Intrusion Detection (Task 2.3)
Tạo ra các chỉ số đặc trưng phân biệt lưu lượng tấn công mạng và IoT:
1. **Ratio Features (Tỉ lệ kích thước gói & tốc độ)**:
   - `avg_packet_size`: Kích thước trung bình gói ($\text{Tot sum} / \text{Number}$)
   - `rate_ratio`: Tỉ lệ tốc độ nguồn ($\text{Srate} / \text{Rate}$)
   - `drate_ratio`: Tỉ lệ tốc độ đích ($\text{Drate} / \text{Rate}$)
   - `duration_ratio`: Tỉ lệ thời gian phiên ($\text{Duration} / \text{flow\_duration}$)
   - `header_to_payload_ratio`: Tỉ lệ header so với dung lượng ($\text{Header\_Length} / \text{Tot size}$)
2. **Flag Aggregates (Tổng hợp cờ TCP)**:
   - `total_flags`: Tổng các cờ điều khiển ($\text{fin} + \text{syn} + \text{rst} + \text{psh} + \text{ack} + \text{ece} + \text{cwr}$)
   - `total_counts`: Tổng các bộ đếm cờ ($\text{ack} + \text{syn} + \text{fin} + \text{urg} + \text{rst}$)
   - `syn_ack_ratio`: Tỉ lệ gói SYN trên ACK (đặc trưng cốt lõi để nhận diện SYN Flood)
   - `rst_syn_ratio`: Tỉ lệ hủy kết nối RST trên SYN
3. **Protocol Groupings (Nhóm giao thức mạng)**:
   - `web_traffic`: HTTP + HTTPS
   - `transport_traffic`: TCP + UDP + ICMP
   - `network_mgmt`: DNS + DHCP + ARP
   - `remote_access`: Telnet + SSH
   - `is_encrypted`: HTTPS + SSH


In [ ]:
# ==============================================================================
# Cell 5: Feature Engineering (Task 2.3)
# ==============================================================================
print("="*60)
print("TASK 2.3: FEATURE ENGINEERING")
print("="*60)

eps = 1e-6  # Chống chia cho 0

# 1. Ratio Features
print("1. Tạo các đặc trưng tỉ lệ (ratios)...")
if 'Tot sum' in df_train.columns and 'Number' in df_train.columns:
    df_train['avg_packet_size'] = (df_train['Tot sum'] / (df_train['Number'] + eps)).astype('float32')

if 'Srate' in df_train.columns and 'Rate' in df_train.columns:
    df_train['rate_ratio'] = (df_train['Srate'] / (df_train['Rate'] + eps)).astype('float32')

if 'Drate' in df_train.columns and 'Rate' in df_train.columns:
    df_train['drate_ratio'] = (df_train['Drate'] / (df_train['Rate'] + eps)).astype('float32')

if 'Duration' in df_train.columns and 'flow_duration' in df_train.columns:
    df_train['duration_ratio'] = (df_train['Duration'] / (df_train['flow_duration'] + eps)).astype('float32')

if 'Header_Length' in df_train.columns and 'Tot size' in df_train.columns:
    df_train['header_to_payload_ratio'] = (df_train['Header_Length'] / (df_train['Tot size'] + eps)).astype('float32')

# 2. Flag Aggregates
print("2. Tạo các đặc trưng tổng hợp TCP Flags...")
flag_number_cols = ['fin_flag_number', 'syn_flag_number', 'rst_flag_number', 
                    'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number']
existing_flag_nums = [c for c in flag_number_cols if c in df_train.columns]
if existing_flag_nums:
    df_train['total_flags'] = df_train[existing_flag_nums].sum(axis=1).astype('float32')

flag_counts_cols = ['ack_count', 'syn_count', 'fin_count', 'urg_count', 'rst_count']
existing_flag_counts = [c for c in flag_counts_cols if c in df_train.columns]
if existing_flag_counts:
    df_train['total_counts'] = df_train[existing_flag_counts].sum(axis=1).astype('float32')

if 'syn_count' in df_train.columns and 'ack_count' in df_train.columns:
    df_train['syn_ack_ratio'] = (df_train['syn_count'] / (df_train['ack_count'] + eps)).astype('float32')

if 'rst_count' in df_train.columns and 'syn_count' in df_train.columns:
    df_train['rst_syn_ratio'] = (df_train['rst_count'] / (df_train['syn_count'] + eps)).astype('float32')

# 3. Protocol Groupings
print("3. Tạo các nhóm giao thức...")
if 'HTTP' in df_train.columns and 'HTTPS' in df_train.columns:
    df_train['web_traffic'] = (df_train['HTTP'] + df_train['HTTPS']).astype('float32')

transport_cols = [c for c in ['TCP', 'UDP', 'ICMP'] if c in df_train.columns]
if transport_cols:
    df_train['transport_traffic'] = df_train[transport_cols].sum(axis=1).astype('float32')

mgmt_cols = [c for c in ['DNS', 'DHCP', 'ARP'] if c in df_train.columns]
if mgmt_cols:
    df_train['network_mgmt'] = df_train[mgmt_cols].sum(axis=1).astype('float32')

remote_cols = [c for c in ['Telnet', 'SSH'] if c in df_train.columns]
if remote_cols:
    df_train['remote_access'] = df_train[remote_cols].sum(axis=1).astype('float32')

if 'HTTPS' in df_train.columns and 'SSH' in df_train.columns:
    df_train['is_encrypted'] = (df_train['HTTPS'] + df_train['SSH']).astype('float32')

# Xử lý các giá trị inf/nan phát sinh từ phép chia 0/0
df_train.replace([np.inf, -np.inf], 0, inplace=True)
df_train.fillna(0, inplace=True)

gc.collect()

all_eng_features = [
    'avg_packet_size', 'rate_ratio', 'drate_ratio', 'duration_ratio',
    'header_to_payload_ratio', 'total_flags', 'total_counts',
    'syn_ack_ratio', 'rst_syn_ratio', 'web_traffic', 'transport_traffic',
    'network_mgmt', 'remote_access', 'is_encrypted'
]
created_eng_features = [c for c in all_eng_features if c in df_train.columns]

print(f"\nĐã tạo thành công {len(created_eng_features)} đặc trưng mới:")
for feat in created_eng_features:
    print(f"  + {feat}")

print(f"\nKích thước tập Train sau khi Feature Engineering: {df_train.shape[0]:,} dòng x {df_train.shape[1]} cột")


## Bước 5: Phân cấp nhãn tấn công (Task 2.4)
CICIoT2023 hỗ trợ 3 cấp độ phân loại bài toán:
1. `label`: 35 lớp chi tiết nguyên bản (Multi-class fine-grained)
2. `attack_category`: 8 nhóm lớp tấn công lớn (`DDoS`, `DoS`, `Mirai`, `Recon`, `Spoofing`, `Web`, `BruteForce`, `Malware`, `Benign`)
3. `is_attack`: Phân loại nhị phân (`0` = Lưu lượng bình thường Benign, `1` = Bị tấn công Attack)


In [ ]:
# ==============================================================================
# Cell 6: Phân cấp nhãn mục tiêu (Task 2.4)
# ==============================================================================
print("="*60)
print("TASK 2.4: PHÂN CẤP NHÃN MỤC TIÊU (HIERARCHICAL LABELS)")
print("="*60)

# Bảng mapping toàn bộ 34 dạng tấn công trong tập CICIoT2023 về 8 nhóm
attack_category_mapping = {
    # 1. DDoS attacks
    'DDoS-ICMP_Flood': 'DDoS',
    'DDoS-UDP_Flood': 'DDoS',
    'DDoS-TCP_Flood': 'DDoS',
    'DDoS-PSHACK_Flood': 'DDoS',
    'DDoS-SYN_Flood': 'DDoS',
    'DDoS-RSTFINFlood': 'DDoS',
    'DDoS-SynonymousIP_Flood': 'DDoS',
    'DDoS-ICMP_Fragmentation': 'DDoS',
    'DDoS-UDP_Fragmentation': 'DDoS',
    'DDoS-ACK_Fragmentation': 'DDoS',
    'DDoS-HTTP_Flood': 'DDoS',
    'DDoS-SlowLoris': 'DDoS',

    # 2. DoS attacks
    'DoS-UDP_Flood': 'DoS',
    'DoS-TCP_Flood': 'DoS',
    'DoS-SYN_Flood': 'DoS',
    'DoS-HTTP_Flood': 'DoS',

    # 3. Mirai attacks
    'Mirai-greeth_flood': 'Mirai',
    'Mirai-udpplain': 'Mirai',
    'Mirai-greip_flood': 'Mirai',

    # 4. Reconnaissance & Scanning
    'Recon-HostDiscovery': 'Recon',
    'Recon-OSScan': 'Recon',
    'Recon-PortScan': 'Recon',
    'Recon-PingSweep': 'Recon',
    'VulnerabilityScan': 'Recon',

    # 5. Spoofing & MITM
    'MITM-ArpSpoofing': 'Spoofing',
    'DNS_Spoofing': 'Spoofing',

    # 6. Web-based attacks
    'BrowserHijacking': 'Web',
    'CommandInjection': 'Web',
    'SqlInjection': 'Web',
    'XSS': 'Web',
    'Uploading_Attack': 'Web',

    # 7. Brute Force
    'DictionaryBruteForce': 'BruteForce',

    # 8. Malware
    'Backdoor_Malware': 'Malware',

    # 9. Benign
    'BenignTraffic': 'Benign'
}

# 1. Tạo cột attack_category
df_train['attack_category'] = df_train['label'].map(lambda x: attack_category_mapping.get(x, 'Other'))

# 2. Tạo nhãn nhị phân: 0 = Benign, 1 = Attack
df_train['is_attack'] = (df_train['label'] != 'BenignTraffic').astype('int8')

print("Phân bố 8 nhóm tấn công trong tập Train:")
category_stats = pd.DataFrame({
    'Số lượng': df_train['attack_category'].value_counts(),
    'Tỉ lệ (%)': (df_train['attack_category'].value_counts(normalize=True) * 100).round(2)
})
print(category_stats)

print("\nPhân bố nhãn nhị phân (is_attack):")
bin_counts = df_train['is_attack'].value_counts()
bin_pct = (df_train['is_attack'].value_counts(normalize=True) * 100).round(2)

bin_stats = pd.DataFrame({
    'Số lượng': [bin_counts.get(0, 0), bin_counts.get(1, 0)],
    'Tỉ lệ (%)': [bin_pct.get(0, 0.0), bin_pct.get(1, 0.0)]
}, index=['Benign (0)', 'Attack (1)'])
print(bin_stats)


## Bước 6: Tách biến đầu vào X_train và mục tiêu y_train (Không chia train/test)
Vì file `train.csv` chỉ dùng thuần túy cho mục đích huấn luyện:
- Tách ma trận đặc trưng $X_{train}$ và vector nhãn mục tiêu $y_{train}$.
- Giữ nguyên toàn bộ dữ liệu của `train.csv` để học các tham số tiền xử lý.
- Giải phóng DataFrame gốc `df_train` khỏi bộ nhớ RAM.


In [ ]:
# ==============================================================================
# Cell 7: Tách biến đặc trưng và nhãn mục tiêu tập Train
# ==============================================================================
print("="*60)
print("TÁCH ĐẶC TRƯNG X_TRAIN VÀ NHÃN Y_TRAIN (TOÀN BỘ TẬP HUẤN LUYỆN)")
print("="*60)

target_columns = ['label', 'attack_category', 'is_attack']
feature_columns = [c for c in df_train.columns if c not in target_columns]

X_train = df_train[feature_columns].copy()
y_train_raw = df_train['label'].copy()
y_cat_train_raw = df_train['attack_category'].copy()
y_bin_train = df_train['is_attack'].copy()

print(f"Ma trận đặc trưng X_train: {X_train.shape[0]:,} dòng x {X_train.shape[1]} đặc trưng")
print(f"Nhãn chi tiết y_train:     {y_train_raw.shape[0]:,} mẫu ({y_train_raw.nunique()} lớp độc nhất)")
print(f"Nhóm nhãn y_cat_train:     {y_cat_train_raw.shape[0]:,} mẫu ({y_cat_train_raw.nunique()} nhóm độc nhất)")

# Giải phóng bộ nhớ RAM
del df_train
gc.collect()
print("Đã giải phóng bộ nhớ df_train thô.")


## Bước 7: Encoding nhãn & Chuẩn hóa đặc trưng RobustScaler (Task 2.5)
1. `LabelEncoder`: Fit trên `y_train_raw` và `y_cat_train_raw` để chuyển nhãn chữ thành số nguyên `0..K-1`.
2. `RobustScaler`: 
   - Trong dữ liệu lưu lượng mạng (network flow), các đợt bùng nổ tấn công tạo ra rất nhiều outliers cực đoan.
   - `RobustScaler` sử dụng Trung vị (`Median`) và Khoảng tứ phân vị ($IQR = Q_3 - Q_1$), giúp mô hình không bị lệch bởi các giá trị ngoại lai so với `StandardScaler`.
   - Scaler được fit trên toàn bộ $X_{train}$ và lưu lại.


In [ ]:
# ==============================================================================
# Cell 8: Label Encoding & Feature Scaling (Task 2.5)
# ==============================================================================
print("="*60)
print("TASK 2.5: LABEL ENCODING & ROBUST FEATURE SCALING")
print("="*60)

# 1. Mã hóa nhãn chi tiết (35 lớp)
le_label = LabelEncoder()
y_train_encoded = le_label.fit_transform(y_train_raw)

# 2. Mã hóa nhóm tấn công (8 nhóm)
le_cat = LabelEncoder()
y_cat_train_encoded = le_cat.fit_transform(y_cat_train_raw)

label_mapping = {label: int(code) for code, label in enumerate(le_label.classes_)}
category_mapping = {cat: int(code) for code, cat in enumerate(le_cat.classes_)}

print(f"Đã mã hóa {len(le_label.classes_)} nhãn chi tiết.")
print(f"Đã mã hóa {len(le_cat.classes_)} nhóm tấn công: {list(category_mapping.keys())}")

# 3. Chuẩn hóa RobustScaler trên X_train
print("\nĐang fit và transform RobustScaler trên X_train...")
scaler = RobustScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
    dtype='float32'
)

print("Chuẩn hóa đặc trưng X_train hoàn tất!")
print(f"Kích thước X_train_scaled: {X_train_scaled.shape}")

# Giải phóng X_train chưa scale
del X_train
gc.collect()


## Bước 8: Lưu trữ Deliverables & Pipeline tiền xử lý
Lưu các thành phần đã được chuẩn bị theo kế hoạch:
- `X_train.pkl`, `y_train.pkl` (lưu cả ở thư mục gốc và `data/`)
- `data/X_train.parquet` (chuẩn lưu trữ nén tốc độ cao)
- `models/preprocessor.joblib`: Đóng gói toàn bộ pipeline gồm:
  - `scaler`: RobustScaler đã fit
  - `label_encoder`: LabelEncoder nhãn chi tiết
  - `category_encoder`: LabelEncoder nhóm tấn công
  - `feature_names`: Danh sách thứ tự các đặc trưng chuẩn
  - `dropped_zero_var_cols`: Danh sách cột hằng số bị loại bỏ
  - `imputation_medians`: Giá trị median của từng cột để điền khuyết
  - `attack_category_mapping`: Bảng ánh xạ 34 loại tấn công về 8 nhóm
- `models/label_mapping.json`: Bảng JSON tra cứu nhãn sang ID


In [ ]:
# ==============================================================================
# Cell 9: Lưu trữ dữ liệu Train và Artifacts Pipeline
# ==============================================================================
print("="*60)
print("LƯU TRỮ DELIVERABLES & PREPROCESSOR PIPELINE")
print("="*60)

# Đóng gói đối tượng nhãn y_train_dict
y_train_dict = {
    'label_encoded': y_train_encoded,
    'label_name': y_train_raw.values,
    'category_encoded': y_cat_train_encoded,
    'category_name': y_cat_train_raw.values,
    'is_attack': y_bin_train.values
}

# 1. Lưu file Pickle
print("1. Lưu file pickle (X_train.pkl, y_train.pkl)...")
joblib.dump(X_train_scaled, 'X_train.pkl', compress=3)
joblib.dump(y_train_dict, 'y_train.pkl', compress=3)

# Đồng thời lưu vào data/ để quản lý gọn gàng
joblib.dump(X_train_scaled, 'data/X_train.pkl', compress=3)
joblib.dump(y_train_dict, 'data/y_train.pkl', compress=3)

# 2. Lưu định dạng Parquet (đọc siêu nhanh, tiết kiệm RAM)
print("2. Lưu parquet (data/X_train.parquet)...")
try:
    X_train_scaled.to_parquet('data/X_train.parquet', index=False)
    print("  -> Đã lưu data/X_train.parquet thành công.")
except Exception as e:
    print(f"  -> Bỏ qua parquet: {e}")

# 3. Đóng gói Preprocessor Pipeline hoàn chỉnh
print("3. Lưu pipeline tiền xử lý vào models/preprocessor.joblib...")
preprocessor = {
    'scaler': scaler,
    'feature_names': list(X_train_scaled.columns),
    'dropped_zero_var_cols': dropped_zero_var_cols,
    'imputation_medians': imputation_medians,
    'label_encoder': le_label,
    'category_encoder': le_cat,
    'label_mapping': label_mapping,
    'category_mapping': category_mapping,
    'attack_category_mapping': attack_category_mapping,
    'engineered_features': created_eng_features
}
joblib.dump(preprocessor, 'models/preprocessor.joblib')

# 4. Lưu file JSON ánh xạ nhãn
with open('models/label_mapping.json', 'w') as f:
    json.dump({
        'fine_grained_labels': label_mapping,
        'attack_categories': category_mapping
    }, f, indent=4)

print("\nCác file đã được tạo thành công:")
for fname in ['X_train.pkl', 'y_train.pkl', 'data/X_train.parquet', 'models/preprocessor.joblib', 'models/label_mapping.json']:
    if os.path.exists(fname):
        size_mb = os.path.getsize(fname) / 1024**2
        print(f"  - {fname}: {size_mb:.2f} MB")


## Bước 9: Hàm tiền xử lý cho tập `validation.csv` & `test.csv` (Khi nhận được sau này)
Khi bạn nhận được file `validation.csv` hoặc `test.csv`, bạn chỉ cần gọi hàm `transform_val_or_test()` bên dưới:
- Tự động áp dụng cùng một bộ lọc đặc trưng, cùng công thức Feature Engineering.
- Dùng `imputation_medians` từ tập Train để điền khuyết (không tính lại median trên test/val).
- Dùng `scaler.transform()` (chỉ transform, tuyệt đối không fit lại).
- Đảm bảo tính nhất quán 100% và chuẩn khoa học dữ liệu.


In [ ]:
# ==============================================================================
# Cell 10: Hàm tiện ích tiền xử lý tập Validation/Test trong tương lai
# ==============================================================================
def transform_val_or_test(csv_path, preprocessor_path='models/preprocessor.joblib', sample_size=None):
    '''
    Hàm tiền xử lý tập validation.csv hoặc test.csv dựa trên pipeline đã fit từ train.csv.
    Sử dụng:
        X_test, y_test_dict = transform_val_or_test('test.csv')
        X_val, y_val_dict   = transform_val_or_test('validation.csv')
    '''
    print(f"--- Đang tiền xử lý: {csv_path} ---")
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Không tìm thấy file: {csv_path}")

    # 1. Tải preprocessor đã lưu
    prep = joblib.load(preprocessor_path)
    
    # 2. Đọc file
    df_raw = pd.read_csv(csv_path, dtype=dtype_dict, nrows=sample_size)
    print(f"Đã đọc: {df_raw.shape[0]:,} dòng x {df_raw.shape[1]} cột")

    # 3. Xử lý inf và điền khuyết bằng median của tập Train
    df_raw.replace([np.inf, -np.inf], np.nan, inplace=True)
    for col, med in prep['imputation_medians'].items():
        if col in df_raw.columns:
            df_raw[col].fillna(med, inplace=True)

    # 4. Loại bỏ các cột hằng số giống tập Train
    if prep['dropped_zero_var_cols']:
        df_raw.drop(columns=[c for c in prep['dropped_zero_var_cols'] if c in df_raw.columns], inplace=True)

    # 5. Feature Engineering giống hệt tập Train
    eps = 1e-6
    if 'Tot sum' in df_raw.columns and 'Number' in df_raw.columns:
        df_raw['avg_packet_size'] = (df_raw['Tot sum'] / (df_raw['Number'] + eps)).astype('float32')
    if 'Srate' in df_raw.columns and 'Rate' in df_raw.columns:
        df_raw['rate_ratio'] = (df_raw['Srate'] / (df_raw['Rate'] + eps)).astype('float32')
    if 'Drate' in df_raw.columns and 'Rate' in df_raw.columns:
        df_raw['drate_ratio'] = (df_raw['Drate'] / (df_raw['Rate'] + eps)).astype('float32')
    if 'Duration' in df_raw.columns and 'flow_duration' in df_raw.columns:
        df_raw['duration_ratio'] = (df_raw['Duration'] / (df_raw['flow_duration'] + eps)).astype('float32')
    if 'Header_Length' in df_raw.columns and 'Tot size' in df_raw.columns:
        df_raw['header_to_payload_ratio'] = (df_raw['Header_Length'] / (df_raw['Tot size'] + eps)).astype('float32')

    flag_number_cols = ['fin_flag_number', 'syn_flag_number', 'rst_flag_number', 
                        'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number']
    active_flags = [c for c in flag_number_cols if c in df_raw.columns]
    if active_flags:
        df_raw['total_flags'] = df_raw[active_flags].sum(axis=1).astype('float32')

    flag_counts_cols = ['ack_count', 'syn_count', 'fin_count', 'urg_count', 'rst_count']
    active_counts = [c for c in flag_counts_cols if c in df_raw.columns]
    if active_counts:
        df_raw['total_counts'] = df_raw[active_counts].sum(axis=1).astype('float32')

    if 'syn_count' in df_raw.columns and 'ack_count' in df_raw.columns:
        df_raw['syn_ack_ratio'] = (df_raw['syn_count'] / (df_raw['ack_count'] + eps)).astype('float32')
    if 'rst_count' in df_raw.columns and 'syn_count' in df_raw.columns:
        df_raw['rst_syn_ratio'] = (df_raw['rst_count'] / (df_raw['syn_count'] + eps)).astype('float32')

    if 'HTTP' in df_raw.columns and 'HTTPS' in df_raw.columns:
        df_raw['web_traffic'] = (df_raw['HTTP'] + df_raw['HTTPS']).astype('float32')
    trans_cols = [c for c in ['TCP', 'UDP', 'ICMP'] if c in df_raw.columns]
    if trans_cols:
        df_raw['transport_traffic'] = df_raw[trans_cols].sum(axis=1).astype('float32')
    mgmt_cols = [c for c in ['DNS', 'DHCP', 'ARP'] if c in df_raw.columns]
    if mgmt_cols:
        df_raw['network_mgmt'] = df_raw[mgmt_cols].sum(axis=1).astype('float32')
    remote_cols = [c for c in ['Telnet', 'SSH'] if c in df_raw.columns]
    if remote_cols:
        df_raw['remote_access'] = df_raw[remote_cols].sum(axis=1).astype('float32')
    if 'HTTPS' in df_raw.columns and 'SSH' in df_raw.columns:
        df_raw['is_encrypted'] = (df_raw['HTTPS'] + df_raw['SSH']).astype('float32')

    df_raw.replace([np.inf, -np.inf], 0, inplace=True)
    df_raw.fillna(0, inplace=True)

    # 6. Tách đặc trưng theo đúng thứ tự cột của tập Train
    expected_features = prep['feature_names']
    for f in expected_features:
        if f not in df_raw.columns:
            df_raw[f] = 0.0

    X_unscaled = df_raw[expected_features].copy()
    
    # 7. Scale đặc trưng bằng Scaler đã fit
    X_scaled = pd.DataFrame(
        prep['scaler'].transform(X_unscaled),
        columns=expected_features,
        index=X_unscaled.index,
        dtype='float32'
    )

    # 8. Encode nhãn nếu file có cột label
    y_dict = None
    if 'label' in df_raw.columns:
        cat_series = df_raw['label'].map(lambda x: prep['attack_category_mapping'].get(x, 'Other'))
        bin_series = (df_raw['label'] != 'BenignTraffic').astype('int8')
        
        # Xử lý nhãn chưa từng thấy (unseen labels) nếu có
        known_labels = set(prep['label_encoder'].classes_)
        safe_labels = df_raw['label'].map(lambda x: x if x in known_labels else prep['label_encoder'].classes_[0])
        
        y_dict = {
            'label_encoded': prep['label_encoder'].transform(safe_labels),
            'label_name': df_raw['label'].values,
            'category_encoded': prep['category_encoder'].transform(cat_series),
            'category_name': cat_series.values,
            'is_attack': bin_series.values
        }

    print(f"Tiền xử lý hoàn tất! X_scaled: {X_scaled.shape}")
    return X_scaled, y_dict

print("Đã định nghĩa hàm transform_val_or_test() sẵn sàng sử dụng khi có test.csv / validation.csv.")


## Bước 10: Tổng kết Phase 2 & Sẵn sàng cho Phase 3
Báo cáo kiểm tra chất lượng dữ liệu tập Train và bàn giao cho Phase 3.


In [ ]:
# ==============================================================================
# Cell 11: Báo cáo tổng kết Phase 2 & Hướng dẫn Phase 3
# ==============================================================================
print("="*70)
print("PHASE 2: BÁO CÁO TỔNG KẾT TIỀN XỬ LÝ (TRAINING SET)")
print("="*70)

print(f'''
TỔNG QUAN TẬP TRAIN SAU TIỀN XỬ LÝ
-----------------------------------------------------------------------
   Số lượng mẫu huấn luyện:    {X_train_scaled.shape[0]:,} mẫu (100% từ train.csv)
   Số lượng đặc trưng đầu vào: {X_train_scaled.shape[1]} (gồm đặc trưng gốc + engineered)
   Số lớp chi tiết:            {len(label_mapping)} lớp
   Số nhóm tấn công:           {len(category_mapping)} nhóm cấp cao
   Tập Test/Validation:        Sẽ nạp và transform qua hàm transform_val_or_test() khi nhận được

KIỂM TRA CHẤT LƯỢNG DỮ LIỆU
-----------------------------------------------------------------------
   X_train số lượng NaN:       {X_train_scaled.isnull().sum().sum()} (100% Sạch)
   X_train số lượng Inf:       {np.isinf(X_train_scaled.values).sum()} (100% Sạch)
   Bộ nhớ X_train_scaled:      {X_train_scaled.memory_usage(deep=True).sum() / 1024**2:.2f} MB

CÁC TỆP ĐÃ XUẤT RA
-----------------------------------------------------------------------
   [x] X_train.pkl, y_train.pkl (ở root và data/)
   [x] data/X_train.parquet (chuẩn tối ưu nén nhanh)
   [x] models/preprocessor.joblib (đóng gói pipeline imputer, scaler, encoders)
   [x] models/label_mapping.json (bảng mapping nhãn)

BƯỚC TIẾP THEO (Theo PLAN.md)
-----------------------------------------------------------------------
   -> Phase 3: Class Imbalance Handling (03_balance.py)
   -> Áp dụng xử lý mất cân bằng lớp trên tập X_train và y_train vừa tạo
   -> Chiến lược: Class Weights / SMOTE / Subsampling
''')

print("="*70)
print("PHASE 2 HOÀN TẤT THÀNH CÔNG! BẠN ĐÃ SẴN SÀNG CHUYỂN SANG PHASE 3.")
print("="*70)
